# Predicting `measured_mem_total`

Simple regression model using all non-target columns from a CSV.
Features include `size`, `attempts`, run statistics (`stop_nodes`, `stop_classes`, `stop_time`, `last_*`), `estimated_mem_egraph`, parsed `stop_reason`, and term-derived structural features.

In [ ]:
import re
from pathlib import Path

import altair as alt
import numpy as np
import polars as pl
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Datasets here exceed Altair's 5k-row default guard; render inline.
alt.data_transformers.enable("default", max_rows=None)

CSV_PATH = Path("../output_new.csv")
TARGET = "measured_mem_total"
SEED = 42

In [ ]:
df = pl.read_csv(CSV_PATH)
df.head()

## Feature engineering

- Parse `stop_reason` into `stop_kind` (categorical: `NodeLimit` / `TimeLimit` / ...) and `stop_limit` (the numeric inside the parens).
- Drop `term` and `size`/`attempts` — keep only run statistics that the Rust function will have available.

In [ ]:
def parse_stop(reason: str) -> tuple[str, float]:
    m = re.match(r"([A-Za-z]+)(?:\(([^)]+)\))?", reason)
    if m is None:
        return (reason, 0.0)
    kind = m.group(1)
    raw = m.group(2)
    val = float(raw) if raw is not None else 0.0
    return (kind, val)


rows = df.to_dicts()
for r in rows:
    kind, limit = parse_stop(r["stop_reason"])
    r["stop_kind"] = kind
    r["stop_limit"] = limit

feat_df = pl.DataFrame(rows)
feat_df = feat_df.to_dummies(columns=["stop_kind"])
feat_df.head()

In [ ]:
drop_cols = [
    TARGET,
    "term",
    "stop_reason",
    "size",
    "attempts",
    "last_time",
    "stop_time",
    "stop_kind_NodeLimit",
    "stop_kind_TimeLimit",
    "stop_limit",
]
feature_cols = [c for c in feat_df.columns if c not in drop_cols]

X = feat_df.select(feature_cols).to_numpy().astype(np.float64)
y = feat_df.get_column(TARGET).to_numpy().astype(np.float64)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X.shape, y.shape, X_train.shape, X_test.shape

## Models

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae = mean_absolute_error(y_te, pred)
    r2 = r2_score(y_te, pred)
    print(f"{name:25s}  MAE={mae:.3e}  R2={r2:.4f}")
    return model, pred


linreg = Pipeline([("scale", StandardScaler()), ("lr", LinearRegression())])
ridge = Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
pca_lr = Pipeline(
    [
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=SEED)),
        ("lr", LinearRegression()),
    ]
)

_, lin_pred = evaluate("LinearRegression", linreg, X_train, y_train, X_test, y_test)
_, _ = evaluate("Ridge(alpha=1.0)", ridge, X_train, y_train, X_test, y_test)
pca_fit, _ = evaluate("PCA(0.95) + LinReg", pca_lr, X_train, y_train, X_test, y_test)

n_kept = pca_fit.named_steps["pca"].n_components_
print(f"\nPCA kept {n_kept} components (95% variance) out of {X_train.shape[1]}")

## Feature importance (linear model)

Importance is read off the *standardized* coefficients of the fitted `LinearRegression`. Because every feature has unit variance after the `StandardScaler`, the magnitude of each coefficient is directly comparable: it answers "how much does the prediction move (in target units) when this feature moves by one standard deviation?"

In [ ]:
scaled_coefs = linreg.named_steps["lr"].coef_
order = np.argsort(np.abs(scaled_coefs))[::-1]

print(f"{'feature':30s} {'std_coef':>14s}")
for i in order:
    print(f"{feature_cols[i]:30s} {scaled_coefs[i]:+14.3e}")

imp = pl.DataFrame(
    {
        "feature": [feature_cols[i] for i in order],
        "coef": [float(scaled_coefs[i]) for i in order],
    }
).with_columns(
    pl.col("coef").abs().alias("magnitude"),
    pl.when(pl.col("coef") < 0).then(pl.lit("negative")).otherwise(pl.lit("positive")).alias("sign"),
)

alt.Chart(imp).mark_bar().encode(
    x=alt.X("magnitude:Q", title="|standardized coefficient|  (target units per 1 sd of feature)"),
    y=alt.Y("feature:N", sort=None, title=None),
    # Diverging by sign: two hues, not a ramp (blue = positive, red = negative).
    color=alt.Color(
        "sign:N",
        scale=alt.Scale(domain=["positive", "negative"], range=["#2a78d6", "#e34948"]),
        legend=alt.Legend(title=None),
    ),
    tooltip=["feature:N", alt.Tooltip("coef:Q", format="+.3e")],
).properties(width=560, height=28 * imp.height, title="Linear-model feature importance")

## Predicted vs. actual and residuals (linear model)

In [ ]:
residuals = y_test - lin_pred
pv = pl.DataFrame({"actual": y_test, "predicted": lin_pred, "residual": residuals})

lo = float(min(y_test.min(), lin_pred.min()))
hi = float(max(y_test.max(), lin_pred.max()))
diag = pl.DataFrame({"x": [lo, hi], "y": [lo, hi]})

scatter = alt.Chart(pv).mark_circle(size=45, opacity=0.7, stroke="black", strokeWidth=0.3)

pred_vs_actual = alt.layer(
    alt.Chart(diag).mark_line(strokeDash=[4, 4], color="#555").encode(x="x:Q", y="y:Q"),
    scatter.encode(
        x=alt.X("actual:Q", title="actual measured_mem_total"),
        y=alt.Y("predicted:Q", title="predicted"),
        tooltip=[alt.Tooltip("actual:Q", format=".2e"), alt.Tooltip("predicted:Q", format=".2e")],
    ),
).properties(width=380, height=340, title=f"Predicted vs. actual  (R² = {r2_score(y_test, lin_pred):.4f})")

resid = alt.layer(
    alt.Chart(pl.DataFrame({"y": [0.0]})).mark_rule(strokeDash=[4, 4], color="#555").encode(y="y:Q"),
    scatter.encode(
        x=alt.X("predicted:Q", title="predicted"),
        y=alt.Y("residual:Q", title="residual (actual − predicted)"),
        tooltip=[alt.Tooltip("predicted:Q", format=".2e"), alt.Tooltip("residual:Q", format=".2e")],
    ),
).properties(width=380, height=340, title="Residuals vs. predicted")

pred_vs_actual | resid

## Feature/target correlation heatmap

The bottom row / right column shows correlation of each feature with the target. Off-diagonal entries reveal feature collinearity (pairs that move together hurt linear-model coefficient interpretability).

In [ ]:
corr_data = np.column_stack([X, y])
corr_labels = feature_cols + [TARGET]
corr = np.corrcoef(corr_data, rowvar=False)

# Long form: one row per (row-label, col-label) cell, preserving label order.
cells = pl.DataFrame(
    {
        "row": [corr_labels[i] for i in range(len(corr_labels)) for _ in corr_labels],
        "col": [corr_labels[j] for _ in corr_labels for j in range(len(corr_labels))],
        "corr": [float(corr[i, j]) for i in range(len(corr_labels)) for j in range(len(corr_labels))],
    }
)

base = alt.Chart(cells).encode(
    x=alt.X("col:N", sort=corr_labels, title=None, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("row:N", sort=corr_labels, title=None),
)
heat = base.mark_rect().encode(
    # Diverging: red↔blue with a neutral midpoint at 0.
    color=alt.Color(
        "corr:Q",
        scale=alt.Scale(scheme="redblue", domain=[-1, 1], reverse=True),
        legend=alt.Legend(title="Pearson r"),
    ),
    tooltip=["row:N", "col:N", alt.Tooltip("corr:Q", format=".2f")],
)
labels = base.mark_text(fontSize=8).encode(
    text=alt.Text("corr:Q", format=".2f"),
    color=alt.condition(alt.datum.corr > 0.5, alt.value("white"), alt.value("black")),
)
(heat + labels).properties(
    width=460, height=460, title="Pearson correlation: features and target"
)

## PCA explained-variance curve and component sweep

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

pca_full = PCA(random_state=SEED).fit(X_train_s)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
print("cumulative explained variance per #components:")
for i, v in enumerate(cum_var, start=1):
    print(f"  {i:3d}: {v:.4f}")

ks = [k for k in [1, 2, 3, 5, 8, 12, 16, X_train.shape[1]] if k <= X_train.shape[1]]
sweep_r2 = []
sweep_mae = []
print("\nLinReg on first k principal components:")
for k in ks:
    pipe = Pipeline([("pca", PCA(n_components=k, random_state=SEED)), ("lr", LinearRegression())])
    pipe.fit(X_train_s, y_train)
    pred = pipe.predict(X_test_s)
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    sweep_r2.append(r2)
    sweep_mae.append(mae)
    print(f"  k={k:3d}  MAE={mae:.3e}  R2={r2:.4f}")

var_df = pl.DataFrame({"n": np.arange(1, len(cum_var) + 1), "cum_var": cum_var})
var_curve = alt.layer(
    alt.Chart(pl.DataFrame({"y": [0.95]}))
    .mark_rule(color="#e34948", strokeDash=[4, 4])
    .encode(y="y:Q"),
    alt.Chart(var_df)
    .mark_line(point=True, color="#2a78d6")
    .encode(
        x=alt.X("n:Q", title="# principal components"),
        y=alt.Y("cum_var:Q", title="cumulative explained variance", scale=alt.Scale(domain=[0, 1.02])),
        tooltip=["n:Q", alt.Tooltip("cum_var:Q", format=".4f")],
    ),
).properties(width=380, height=300, title="PCA explained-variance curve (red = 95%)")

# Two measures at different scales → two panels, never one dual-axis chart.
sweep_df = pl.DataFrame({"k": ks, "R²": sweep_r2, "MAE": sweep_mae})
r2_panel = (
    alt.Chart(sweep_df)
    .mark_line(point=True, color="#008300")
    .encode(
        x=alt.X("k:Q", title="# components used in regression"),
        y=alt.Y("R²:Q", title="R² (test)"),
        tooltip=["k:Q", alt.Tooltip("R²:Q", format=".4f")],
    )
    .properties(width=180, height=140, title="R² vs # components")
)
mae_panel = (
    alt.Chart(sweep_df)
    .mark_line(point=True, color="#eb6834")
    .encode(
        x=alt.X("k:Q", title="# components used in regression"),
        y=alt.Y("MAE:Q", title="MAE (test)"),
        tooltip=["k:Q", alt.Tooltip("MAE:Q", format=".3e")],
    )
    .properties(width=180, height=140, title="MAE vs # components")
)

var_curve | (r2_panel & mae_panel)

## Export linear model as Rust constants

Folds the `StandardScaler` into the coefficients so the Rust function can use raw feature values:

`y = intercept_raw + sum(coef_raw[i] * x_raw[i])` where
`coef_raw[i] = coef[i] / scale[i]` and
`intercept_raw = intercept - sum(coef[i] * mean[i] / scale[i])`.

In [ ]:
scaler = linreg.named_steps["scale"]
lr = linreg.named_steps["lr"]

mean = scaler.mean_
scale = scaler.scale_
coef_scaled = lr.coef_
intercept_scaled = lr.intercept_

coef_raw = coef_scaled / scale
intercept_raw = intercept_scaled - float(np.sum(coef_scaled * mean / scale))

# sanity-check: predictions match the sklearn pipeline
manual = X_test @ coef_raw + intercept_raw
sk = linreg.predict(X_test)
print(f"max |manual - sklearn| = {np.max(np.abs(manual - sk)):.3e}")

print("\nfeatures (in order):")
for i, name in enumerate(feature_cols):
    print(f"  {i:2d}  {name:25s}  coef={coef_raw[i]:+.6e}")
print(f"\nintercept = {intercept_raw:+.6e}")

In [ ]:
def fmt_rust(x: float) -> str:
    s = f"{x:+.10e}"
    mantissa, exp = s.split("e")
    sign = mantissa[0] if mantissa[0] == "-" else " "
    digits = mantissa[1:]
    int_part, frac_part = digits.split(".")
    # group fractional digits into chunks of 3 from the left for readability
    frac_chunks = [frac_part[i : i + 3] for i in range(0, len(frac_part), 3)]
    frac_grouped = "_".join(frac_chunks)
    return f"{sign}{int_part}.{frac_grouped}e{exp}"


print(f"const INTERCEPT: f64 = {fmt_rust(intercept_raw)};")
print(f"const COEFS: [f64; {len(feature_cols)}] = [")
for name, c in zip(feature_cols, coef_raw):
    print(f"    {fmt_rust(c)}, // {name}")
print("];")
print()
print("features in order:")
for name in feature_cols:
    print(f"  {name}")